In [1]:
import sys
sys.dont_write_bytecode = True

import warnings
warnings.filterwarnings("ignore")

import importlib
from src.main.python.schema import model
importlib.reload(model)

<module 'src.main.python.schema.model' from 'd:\\dev_space\\LLM-Server\\src\\main\\python\\schema\\model.py'>

# unittest

In [6]:
import sys
import subprocess
!python -B -m unittest src.unittest.python.test_LLMserver

.
----------------------------------------------------------------------
Ran 1 test in 0.001s

OK


# dev

In [1]:
import sys
sys.dont_write_bytecode = True

from src.main.python.engine import decode, prefill
from src.main.python.scheduler import scheduler
from src.main.python.config import config
from src.main.python.schema import model
import importlib

import torch
from transformers import GemmaTokenizerFast, BitsAndBytesConfig, Gemma3ForCausalLM, DynamicCache
# PATH = "D://LLM//gemma//gemma3_4b"
PATH = "D://LLM//small_gemma//gemma3_270M"

quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

llm_model = Gemma3ForCausalLM.from_pretrained(
    PATH,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True
    )
llm_model = llm_model.eval()
tokenizer = GemmaTokenizerFast.from_pretrained(PATH)

In [2]:
import uuid
_scheduler = scheduler.RequestManager()
for sentences in ("什麼是機器學習 ?", "半導體廠務通常在做什麼 ?", "什麼是巨單交易 ?", "說明Black Scholes的核心精神，以及詳細解釋他的計算細節與原理 !"):
    ids = tokenizer.encode(sentences)
    request = model.Request(input_ids = torch.tensor(ids).unsqueeze(0),
                            status = model.RequestStatus.PREFILLING,
                            request_id = uuid.uuid4(),
                            kv_cache = DynamicCache()
            ) 
    _scheduler.add_request(request)
d_input_ids, d_caches, p_input_ids, p_caches = _scheduler.step()

In [3]:
decode.infer(llm_model, d_input_ids, d_caches)
prefill.infer(llm_model, p_input_ids, p_caches)

`cache.key_cache[idx]` is deprecated and will be removed in v4.56.0. Use `cache.layers[idx].keys` instead.


TypeError: expected TensorOptions(dtype=float, device=cpu, layout=Strided, requires_grad=false (default), pinned_memory=false (default), memory_format=(nullopt)) (got TensorOptions(dtype=__int64, device=cpu, layout=Strided, requires_grad=false (default), pinned_memory=false (default), memory_format=(nullopt)))

# test

In [8]:
msg = """<start_of_turn>user
[所有回應一律用繁體中文回答]{prompt}<end_of_turn>
<start_of_turn>model
"""

def gemma3_resp(prompt):
    max_seq_len = 32768

    # ----- 結果儲存 ----- #
    res = list()

    # ----- Prompt token產生 ----- #
    MSG = msg.format(prompt=prompt)
    input_ids = torch.tensor(tokenizer.encode(MSG)).to(llm_model.device)
    input_ids = input_ids.unsqueeze(0)
    eos_token_ids = [tokenizer.eos_token_id, 106]

    # ----- Cache宣告 ----- #
    past_key_values = DynamicCache()
    
    # ----- Prefill ----- #
    chunks = torch.split(input_ids[:, :-1], 32, dim=-1)
    st = 0
    ed = 0
    with torch.no_grad():
        for chunk in chunks:
            ed = st + chunk.shape[1]
            llm_model(input_ids=chunk, use_cache=True, past_key_values=past_key_values)
            st = ed
    
    # ----- Auto Regressive生成 ----- #
    input_ids = input_ids[:, -1:]
    attention_mask = torch.ones(1, ed, dtype=torch.long, device=llm_model.device)
    try:
        for _ in range(max_seq_len):
            with torch.no_grad():
                # ----- Update position ----- #
                ed += 1

                # ----- Update model kwargs ----- #
                cache_position = torch.arange(ed-1, ed, dtype=torch.long, device = llm_model.device)

                # ----- 生成token ----- #
                outputs = llm_model(input_ids=input_ids, use_cache=True, past_key_values=past_key_values, cache_position=cache_position)
                logits = outputs.logits
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                token_id = next_token.item()
                input_ids = next_token

                # ----- 判斷是否終止 ----- #
                if token_id in eos_token_ids:
                    break

                # ----- 紀錄token ----- #
                res += [tokenizer.decode(token_id)]
                
                # ----- 輸出文字字串 ----- #
                print(res[-1], end="", flush=True)
    except:
        for item in ("input_ids", "outputs", "ogits", "next_token", "token_id"):
            try:
                eval(f"del {item}")
            except:
                pass
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
    return "".join(res), past_key_values

In [16]:
_, cache1 = gemma3_resp("你好")

好的，謝謝。我會盡我的力量回答你的問題。

In [15]:
_, cache2 = gemma3_resp("什麼是AI ? 一句話介紹一下")

AI (Artificial Intelligence) 是一種人工智能 (Artificial Intelligence) 的一種方法，它通過使用计算机 (c-a-I) 模拟人类的智能，从而学习和模仿人类的语言、动作、学习和推理能力。

AI 是一種高度智能的機器學習 (Machine Learning) 系統，它通过收集和分析大量数据，并使用机器学习 (Machine Learning) 算法来识别和预测这些数据，并根据这些预测结果，自动调整和优化其训练数据，从而提高其准确性和效率。

AI 是一種重要的發展方向，它正在以前存留的科技成果，为人类社会带来新的机遇，并为人们提供新的服务。


In [17]:
import torch
import numpy as np
import torch.nn.functional as F
from typing import List, Dict, Any
from transformers import DynamicCache

def KVCache_merge(caches: List[DynamicCache]):
    results = DynamicCache()
    # ----- 檢查是否全部都為空 ----- #
    empty_check = [c.key_cache[0] is None for c in caches]
    if np.all(empty_check):
        return results

    # ----- 取出layer ----- #
    seq_len = max(c.get_seq_length(layer_idx=0) for c, is_empty in zip(caches, empty_check) if not is_empty)
    first_non_empty_cache = next(c for c, is_empty in zip(caches, empty_check) if not is_empty)
    n_layers = len(first_non_empty_cache.key_cache)
    n_heads, hid_dim = first_non_empty_cache.key_cache[0].shape[1], first_non_empty_cache.key_cache[0].shape[3]
    
    # ----- 建立cache ----- #
    for i in range(n_layers):
        # ----- 依照不同layer去建立cache ----- #
        keys, values = list(), list()
        for c in caches:
            if c.key_cache[0] is None:
                # ----- 如果是空的，則全部補0 ----- #
                key_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.float32)
                value_tensor = torch.zeros((1, n_heads, seq_len, hid_dim), dtype=torch.float32)
                keys += [key_tensor]
                values += [value_tensor]
                continue
            key_tensor = c.key_cache[i]
            value_tensor = c.value_cache[i]

            # ----- 過長的部分做padding ----- # 
            curr_seq_len = key_tensor.shape[2]
            if curr_seq_len < seq_len:
                padding_to_add = seq_len - curr_seq_len
                key_tensor = F.pad(key_tensor, (0, 0, padding_to_add, 0), "constant", 0)
                value_tensor = F.pad(value_tensor, (0, 0, padding_to_add, 0), "constant", 0)

            keys += [key_tensor]
            values += [value_tensor]
        
        # ----- merge tensor ----- #
        key_batch = torch.cat(keys, dim=0)
        value_batch = torch.cat(values, dim=0)

        # ----- update cache ----- #
        results.update(key_states=key_batch, value_states=value_batch, layer_idx=i)
    results.seen_tokens = seq_len
    return results

In [20]:
CACHE = KVCache_merge([cache1, cache2])

In [30]:
CACHE.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],
          [ 0.0000e+00],


In [ ]:
def KVCache_split(cache: DynamicCache):
    # ----- 把cache的layer跟數量定義出來 ----- #
    batch_size = cache.key_cache[0].shape[0]
    n_layers = len(cache.key_cache)

    # ----- return的結果 ----- #
    results: List[DynamicCache] = [DynamicCache() for _ in range(batch_size)]

    # ----- by batch操作
    for i in range(batch_size):
        # ----- cache的原始長度，只要用第0層來找即可 ----- #
        """
        因為padding是用0填充，所以如果 hid_dim 和 n_head 都是0，那該位必定padding
        最終找到最後一個非零位置
        """
        sample_key_tensor = cache.key_cache[0][i:i+1] # (1, n_heads, seq_len, hid_dim)
        sum_abs = torch.abs(sample_key_tensor).sum(dim=(1, 3)).squeeze(0)
        non_zero_indices = torch.where(sum_abs > 1e-6)[0] # (seq_len, )

        # ----- seq_len 儲存長度計算 ----- #
        if len(non_zero_indices) == 0: original_seq_len = 0
        else: original_seq_len = non_zero_indices.min().item()
            
        for layer_idx in range(n_layers):
            key_slice = cache.key_cache[layer_idx][i:i+1]
            value_slice = cache.value_cache[layer_idx][i:i+1]

            truncated_key = key_slice[:, :, original_seq_len:, :]
            truncated_value = value_slice[:, :, original_seq_len:, :]
            
            results[i].update(
                key_states=truncated_key,
                value_states=truncated_value,
                layer_idx=layer_idx
            )
        
        # 6. 更新這個 cache 的 seen_tokens
        results[i].seen_tokens = original_seq_len

    return results

In [33]:
_cache1, _cache2 = KVCache_split(CACHE)
_cache1.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0.0000],
          [ 0

In [34]:
cache1.key_cache[0][:, :, :, 0:1]

tensor([[[[ 0.0347],
          [-2.8125],
          [-3.2812],
          [-0.4941],
          [ 0.1016],
          [ 1.5000],
          [ 0.8594],
          [-4.0625],
          [-3.6094],
          [-1.4531],
          [ 3.5625],
          [ 4.5312],
          [ 1.5156],
          [-1.7109],
          [-0.5078],
          [-2.1250],
          [-0.5391],
          [-0.0435],
          [ 2.9062],
          [-0.9453],
          [ 0.1157],
          [-0.4453],
          [-0.0967],
          [ 2.5156],
          [ 0.2139],
          [ 0.7148],
          [-2.4219],
          [-3.1719],
          [-1.1016],
          [ 1.7188],
          [ 3.3906],
          [ 1.2656],
          [-1.9922],
          [-0.1641]]]], device='cuda:0', dtype=torch.bfloat16)